# Cold-warm trend: plots

_Split from `cw_mean_trend.ipynb`. Shared helpers live in `functions.py`._


In [ ]:
# Load Packages 

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import math
import cartopy.crs as ccrs
import calendar
from matplotlib.lines import Line2D
import os
import re
import seaborn as sns
import pandas as pd
import matplotlib.gridspec as gridspec
from scipy.stats import norm
# from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patches as patches
from scipy.stats import scoreatpercentile
# import xesmf as xe
from matplotlib.gridspec import GridSpec
import matplotlib
from pathlib import Path



In [ ]:
import sys, os
# make this repo's functions.py importable from any subfolder
_p = os.getcwd()
for _ in range(6):
    if os.path.exists(os.path.join(_p, 'functions.py')):
        sys.path.insert(0, _p); break
    _p = os.path.dirname(_p)
from functions import *  # shared helper functions


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

sources = ['ERA5','JRA55-do','uas & vas trend','uas & vas mean & trend','ERA5 winds','rsds & rlds trend','huss & tas trend','all-ish trends','NOAA OISST']
colors  = ['darkviolet','firebrick','cyan','b','violet','orange','limegreen','red','k']
cmap = dict(zip(sources, colors))


keymap = {
    "ERA5": "era5_percentiles",
    "JRA55-do": "jra55_percentiles",
    "uas & vas trend": "uas_vas_trend_percentiles",
    "uas & vas mean & trend": "uas_vas_mean_trend_percentiles",
    "ERA5 winds": "era5_winds_percentiles",
    "rsds & rlds trend": "rsds_rlds_trend_percentiles",
    "huss & tas trend": "huss_tas_trend_percentiles",
    "all-ish trends": "all_trend_percentiles",
    "NOAA OISST": "noaa",
}

cold_means = dict(cold_means)
cold_means["noaa"] = noaa_cold50_mean

cold50_mean_da = xr.DataArray(
    np.array([np.asarray(cold_means[keymap[s]].values).squeeze().item() for s in sources]),
    dims="source",
    coords={"source": sources},
    name="warm50_mean",
)
def _get(d, k):
    if k in d: return d[k]
    if k.lower() in d: return d[k.lower()]
    if k.upper() in d: return d[k.upper()]
    raise KeyError(f"{k} not found. Available keys (sample): {list(d.keys())[:25]}")

cold50_trend_da = xr.DataArray(
    np.array([
        cold50_era5.values,
        cold50_jra55.values,
        cold50_perturb_uas_vas.values,
        cold_50_perturb_winds_mean_trend.values,
        cold50_perturb_winds.values,
        cold50_perturb_rsds_rlds.values,
        cold50_perturb_huss_tas.values,
        cold50_perturb_all.values,
        cold50_noaa.values,
    ]) * 10 * 12,
    dims="source",
    coords={"source": sources},
    name="cold50_trend",
)

percentiles_omip_cold50_mean = {}
percentiles_omip_cold50_trend = {}
for m in mod_labs:
    percentiles_omip_cold50_trend[m] = percentiles_omip_trend[m].sel(area=slice(0, 0.5)).mean(dim="area")
    percentiles_omip_cold50_mean[m]  = (dict_percentiles_omip[m].sel(area=slice(0, 0.5)).mean(dim="area")).mean(dim="time")

vals_omip_trend = [float(v.values * 10 * 12) for v in percentiles_omip_cold50_trend.values()]
vals_omip_mean  = [float(v.values ) for v in percentiles_omip_cold50_mean.values()]

fig, ax = plt.subplots(figsize=(8.5, 6))

ax.scatter(vals_omip_trend,vals_omip_mean, alpha=0.5, s=90, label="OMIP")

for s, c in zip(sources, colors):
    ax.scatter(
        cold50_trend_da.sel(source=s).item(),
        cold50_mean_da.sel(source=s).item(),
        color=c,
        s=90,
        linewidth=0.6,
        label=s,
        zorder=5,
    )

for s in ["ERA5", "JRA55-do", "NOAA OISST"]:
    y0 = cold50_mean_da.sel(source=s).item()
    x0 = cold50_trend_da.sel(source=s).item()
    ax.axvline(x0, linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
    ax.axhline(y0, linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)

ax.set_ylabel(r"Mean of Coldest 50% ($^{\circ}$C 10yr$^{-1}$)")
ax.set_xlabel(r"Trend of Coldest 50% ($^{\circ}$C )")
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

sources = ['ERA5','JRA55-do','uas & vas trend','uas & vas mean & trend','ERA5 winds','rsds & rlds trend','huss & tas trend','all-ish trends','NOAA OISST']
colors  = ['darkviolet','firebrick','cyan','b','violet','orange','limegreen','red','k']
cmap = dict(zip(sources, colors))


keymap = {
    "ERA5": "era5_percentiles",
    "JRA55-do": "jra55_percentiles",
    "uas & vas trend": "uas_vas_trend_percentiles",
    "uas & vas mean & trend": "uas_vas_mean_trend_percentiles",
    "ERA5 winds": "era5_winds_percentiles",
    "rsds & rlds trend": "rsds_rlds_trend_percentiles",
    "huss & tas trend": "huss_tas_trend_percentiles",
    "all-ish trends": "all_trend_percentiles",
    "NOAA OISST": "noaa",
}

warm_means = dict(warm_means)
cold_means = dict(cold_means)
warm_means = dict(warm_means)
warm_means["noaa"] = noaa_warm50_mean

warm50_mean_da = xr.DataArray(
    np.array([np.asarray(warm_means[keymap[s]].values).squeeze().item() for s in sources]),
    dims="source",
    coords={"source": sources},
    name="warm50_mean",
)
def _get(d, k):
    if k in d: return d[k]
    if k.lower() in d: return d[k.lower()]
    if k.upper() in d: return d[k.upper()]
    raise KeyError(f"{k} not found. Available keys (sample): {list(d.keys())[:25]}")

warm50_trend_da = xr.DataArray(
    np.array([
        warm50_era5.values,
        warm50_jra55.values,
        warm50_perturb_uas_vas.values,
        warm50_perturb_winds_mean_trend.values,
        warm50_perturb_winds.values,
        warm50_perturb_rsds_rlds.values,
        warm50_perturb_huss_tas.values,
        warm50_perturb_all.values,
        warm50_noaa.values,
    ]) * 10 * 12,
    dims="source",
    coords={"source": sources},
    name="warm50_trend",
)



percentiles_omip_warm50_mean = {}
percentiles_omip_warm50_trend = {}
for m in mod_labs:
    percentiles_omip_warm50_trend[m] = percentiles_omip_trend[m].sel(area=slice(0.5, 1)).mean(dim="area")
    percentiles_omip_warm50_mean[m]  = (dict_percentiles_omip[m].sel(area=slice(0.5, 1)).mean(dim="area")).mean(dim="time")

vals_omip_trend = [float(v.values * 10 * 12) for v in percentiles_omip_warm50_trend.values()]
vals_omip_mean  = [float(v.values ) for v in percentiles_omip_warm50_mean.values()]

fig, ax = plt.subplots(figsize=(8.5, 6))

ax.scatter(vals_omip_trend,vals_omip_mean, alpha=0.5, s=90, label="OMIP")

for s, c in zip(sources, colors):
    ax.scatter(
        warm50_trend_da.sel(source=s).item(),
        warm50_mean_da.sel(source=s).item(),
        color=c,
        s=90,
        linewidth=0.6,
        label=s,
        zorder=5,
    )

for s in ["ERA5", "JRA55-do", "NOAA OISST"]:
    y0 = warm50_mean_da.sel(source=s).item()
    x0 = warm50_trend_da.sel(source=s).item()
    ax.axvline(x0, linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
    ax.axhline(y0, linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)

ax.set_ylabel(r"Mean of Warmest 50% ($^{\circ}$C 10yr$^{-1}$)")
ax.set_xlabel(r"Trend of Warmest 50% ($^{\circ}$C )")
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
cold50_cmip = DA_all.sel(area=slice(0, 0.5)).mean(dim="area")
warm50_cmip = DA_all.sel(area=slice(0.5, 1)).mean(dim="area")


cold50_cmip_trim = cold50_cmip*10*12 #.where((cold50_cmip >= lo_cold) & (cold50_cmip <= hi_cold), drop=True)*10*12
warm50_cmip_trim = warm50_cmip*10*12 #warm50_cmip.where((warm50_cmip >= lo_warm) & (warm50_cmip <= hi_warm), drop=True)*10*12

# -------------------------
# Obs / perturb definitions
# -------------------------
sources = ['ERA5 forced','JRA55-do forced','JRA55-do forced with ERA5 uas, vas trend','JRA55-do forced with ERA5 uas, vas trend and mean state','JRA55-do forced with ERA5 uas, vas', 'JRA55-do forced with ERA5 rsds, rlds trend','JRA55-do forced with ERA5 huss, tas trend','NOAA OISST']
colors  = ['darkviolet', 'firebrick', 'cyan','b', 'violet', 'limegreen', 'red', 'k']
cmap = dict(zip(sources, colors))

# warm50 obs (same order as sources)
warm50_obs_da = xr.DataArray(
    np.array([
        warm50_era5.values,
        warm50_jra55.values,
        warm50_perturb_uas_vas.values,
        warm50_perturb_winds_mean_trend.values,
        warm50_perturb_winds.values,
        warm50_perturb_rsds_rlds.values,
        warm50_perturb_huss_tas.values,
        # warm50_perturb_all.values,
        warm50_noaa.values
        ]) * 10 * 12,
    dims="source",
    coords={"source": sources},
    name="warm50_trend"
)

# cold50 obs (same order as sources)
cold50_obs_da = xr.DataArray(
    np.array([
        cold50_era5.values,
        cold50_jra55.values,
        cold50_perturb_uas_vas.values,
        cold_50_perturb_winds_mean_trend,
        cold50_perturb_winds.values,
        cold50_perturb_rsds_rlds.values,
        cold50_perturb_huss_tas.values,
        # cold50_perturb_all.values,
        cold50_noaa.values
        ]) * 10 * 12,
    dims="source",
    coords={"source": sources},
    name="cold50_trend"
)

# coldest 50% mean only
cold50_omip = {}
for da in mod_labs:
    cold50_omip[da] = percentiles_omip_trend[da].sel(area=slice(0, 0.5)).mean(dim="area")
vals_cold50 = [float(v.values * 10 * 12) for v in cold50_omip.values()]

warm50_omip = {}
for da in mod_labs:
    warm50_omip[da] = percentiles_omip_trend[da].sel(area=slice(0.5, 1)).mean(dim="area")

vals_warm50 = [float(v.values * 10 * 12) for v in warm50_omip.values()]

# vals
fig, ax = plt.subplots(figsize=(12, 6))

# CMIP cloud
ax.scatter(cold50_cmip_trim, warm50_cmip_trim, color='red', alpha=0.15, label='CMIP')

cmip_xmean = np.nanmean(warm50_cmip_trim)
cmip_ymean = np.nanmean(cold50_cmip_trim)
ax.scatter(cmip_ymean, cmip_xmean, color='indianred', edgecolors='k', s=140,  linewidth=0.8,
           zorder=6, label='CMIP mean')

ax.scatter(vals_cold50,vals_warm50, color='slateblue',alpha=0.5, s=90,label='OMIP')

# Obs / perturb points
for s, c in zip(sources, colors):
    ax.scatter(
        cold50_obs_da.sel(source=s).item(),
        warm50_obs_da.sel(source=s).item(),
        color=c,
        s=90,
        linewidth=0.6,
        label=s,
        zorder=5
    )

for s in ["ERA5 forced", "JRA55-do forced", "NOAA OISST"]:
    y0 = cold50_obs_da.sel(source=s).item()
    x0 = warm50_obs_da.sel(source=s).item()
    ax.axvline(x0, linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
    ax.axhline(y0, linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)

ylabel = r'Trend of Warmest 50% (°C 10 yr$^{-1}$)'
xlabel = r'Trend of Coldest 50% (°C 10 yr$^{-1}$)'
ax.set_xlabel(ylabel)
ax.set_ylabel(xlabel)
# ax.grid()
ax.plot([-1,1.25],[-1,1.25], linestyle='--', color='k', label = '1-1 ratio')
ax.set_xlim([-0.1, 0.4])
ax.set_ylim([-0.05, 0.45])
# legend on the side
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
plt.show()

outpath = "/Users/isabelaconde/Desktop/cold50_warm50_1982-2017.pdf"
fig.savefig(outpath, format="pdf", bbox_inches="tight")



In [ ]:
warm50_jra55 = control_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
cold50_jra55 = control_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")

warm50_era5  = era5_percent_trend.sel(area=slice(0.5, 1)).mean(dim="area")
cold50_era5  = era5_percent_trend.sel(area=slice(0, 0.5)).mean(dim="area")

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

cold50_cmip = DA_all.sel(area=slice(0, 0.5)).mean(dim="area")
warm50_cmip = DA_all.sel(area=slice(0.5, 1)).mean(dim="area")
cold50_cmip_trim = cold50_cmip * 10 * 12
warm50_cmip_trim = warm50_cmip * 10 * 12

sources = ['ERA5','JRA55-do','uas & vas trend','uas & vas mean & trend','ERA5 winds','rsds & rlds trend','huss & tas trend','all-ish trends','NOAA OISST']
colors  = ['darkviolet','firebrick','cyan','b','violet','orange','limegreen','red','k']
cmap = dict(zip(sources, colors))

warm50_obs_da = xr.DataArray(
    np.array([
        warm50_era5.values,
        warm50_jra55.values,
        warm50_perturb_uas_vas.values,
        warm50_perturb_winds_mean_trend.values,
        warm50_perturb_winds.values,
        warm50_perturb_rsds_rlds.values,
        warm50_perturb_huss_tas.values,
        warm50_perturb_all.values,
        warm50_noaa.values
    ]) * 10 * 12,
    dims="source",
    coords={"source": sources},
    name="warm50_trend",
)

cold50_obs_da = xr.DataArray(
    np.array([
        cold50_era5.values,
        cold50_jra55.values,
        cold50_perturb_uas_vas.values,
        cold_50_perturb_winds_mean_trend.values,
        cold50_perturb_winds.values,
        cold50_perturb_rsds_rlds.values,
        cold50_perturb_huss_tas.values,
        cold50_perturb_all.values,
        cold50_noaa.values
    ]) * 10 * 12,
    dims="source",
    coords={"source": sources},
    name="cold50_trend",
)

keymap = {
    "ERA5": "era5_percentiles",
    "JRA55-do": "jra55_percentiles",
    "uas & vas trend": "uas_vas_trend_percentiles",
    "uas & vas mean & trend": "uas_vas_mean_trend_percentiles",
    "ERA5 winds": "era5_winds_percentiles",
    "rsds & rlds trend": "rsds_rlds_trend_percentiles",
    "huss & tas trend": "huss_tas_trend_percentiles",
    "all-ish trends": "all_trend_percentiles",
    "NOAA OISST": "noaa",
}

warm_means = dict(warm_means)
cold_means = dict(cold_means)
warm_means["noaa"] = noaa_warm50_mean
cold_means["noaa"] = noaa_cold50_mean

warm50_mean_da = xr.DataArray(
    np.array([np.asarray(warm_means[keymap[s]].values).squeeze().item() for s in sources]),
    dims="source",
    coords={"source": sources},
    name="warm50_mean",
)

cold50_mean_da = xr.DataArray(
    np.array([np.asarray(cold_means[keymap[s]].values).squeeze().item() for s in sources]),
    dims="source",
    coords={"source": sources},
    name="cold50_mean",
)

vals_warm50_omip = [float(percentiles_omip_trend[m].sel(area=slice(0.5, 1)).mean(dim="area").values * 10 * 12) for m in mod_labs]
vals_cold50_omip = [float(percentiles_omip_trend[m].sel(area=slice(0, 0.5)).mean(dim="area").values * 10 * 12) for m in mod_labs]

vals_warm50_omip_mean = [float(((dict_percentiles_omip[m].sel(area=slice(0.5, 1)).mean(dim="area")).mean(dim="time")).values) for m in mod_labs]
vals_cold50_omip_mean = [float(((dict_percentiles_omip[m].sel(area=slice(0, 0.5)).mean(dim="area")).mean(dim="time")).values) for m in mod_labs]

fig = plt.figure(figsize=(22, 5.5))
gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 0.55], wspace=0.25)

ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1])
ax2 = fig.add_subplot(gs[0, 2])
axL = fig.add_subplot(gs[0, 3])
axL.axis("off")

ax0.set_xlabel("Trend of Coldest 50% (°C/10yr)")
ax0.set_ylabel("Trend of Warmest 50% (°C/10yr)")

ax0.scatter(cold50_cmip_trim, warm50_cmip_trim, color="red", alpha=0.15)
ax0.scatter(np.nanmean(cold50_cmip_trim), np.nanmean(warm50_cmip_trim),
            color="indianred", edgecolors="k", s=140, linewidth=0.8, zorder=6)
ax0.scatter(vals_cold50_omip, vals_warm50_omip, color="slateblue", alpha=0.5, s=90)

for s, c in zip(sources, colors):
    ax0.scatter(cold50_obs_da.sel(source=s).item(), warm50_obs_da.sel(source=s).item(),
                color=c, s=90, linewidth=0.6, zorder=5)

for s in ["ERA5", "JRA55-do", "NOAA OISST"]:
    ax0.axvline(cold50_obs_da.sel(source=s).item(), linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
    ax0.axhline(warm50_obs_da.sel(source=s).item(), linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)

ax0.plot([-1, 1.25], [-1, 1.25], linestyle="--", color="k")
ax0.set_xlim([-0.1, 0.4])
ax0.set_ylim([-0.05, 0.45])
ax0.set_xlabel("Trend of Warmest 50% (°C/10yr)")
ax0.set_ylabel("Trend of Coldest 50% (°C/10yr)")

ax1.scatter(vals_cold50_omip, vals_cold50_omip_mean, color="slateblue", alpha=0.5, s=90)
for s, c in zip(sources, colors):
    ax1.scatter(cold50_obs_da.sel(source=s).item(), cold50_mean_da.sel(source=s).item(),
                color=c, s=90, linewidth=0.6, zorder=5)
for s in ["ERA5", "JRA55-do", "NOAA OISST"]:
    ax1.axvline(cold50_obs_da.sel(source=s).item(), linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
    ax1.axhline(cold50_mean_da.sel(source=s).item(), linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
ax1.set_xlabel(r"Trend of Coldest 50% ($^{\circ}$C/10yr)")
ax1.set_ylabel(r"Mean of Coldest 50% ($^{\circ}$C)")

ax2.scatter(vals_warm50_omip, vals_warm50_omip_mean, color="slateblue", alpha=0.5, s=90)
for s, c in zip(sources, colors):
    ax2.scatter(warm50_obs_da.sel(source=s).item(), warm50_mean_da.sel(source=s).item(),
                color=c, s=90, linewidth=0.6, zorder=5)
for s in ["ERA5", "JRA55-do", "NOAA OISST"]:
    ax2.axvline(warm50_obs_da.sel(source=s).item(), linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
    ax2.axhline(warm50_mean_da.sel(source=s).item(), linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
ax2.set_xlabel(r"Trend of Warmest 50% ($^{\circ}$C/10yr)")
ax2.set_ylabel(r"Mean of Warmest 50% ($^{\circ}$C)")

handles0 = [
    Line2D([0], [0], marker="o", linestyle="None", markersize=8,
           markerfacecolor="red", markeredgecolor="none", alpha=0.15, label="CMIP"),
    Line2D([0], [0], marker="o", linestyle="None", markersize=10,
           markerfacecolor="indianred", markeredgecolor="k", label="CMIP mean"),
    Line2D([0], [0], marker="o", linestyle="None", markersize=8,
           markerfacecolor="slateblue", markeredgecolor="none", alpha=0.5, label="OMIP"),
    Line2D([0], [0], linestyle="--", color="k", label="1–1 ratio"),
]
handles_sources = [
    Line2D([0], [0], marker="o", linestyle="None", markersize=8,
           markerfacecolor=c, markeredgecolor="none", label=s)
    for s, c in zip(sources, colors)
]

axL.legend(handles=handles0 + handles_sources, frameon=False, loc="center left")

plt.show()
out = "/Users/isabelaconde/Desktop/warm_cold_threepanel.pdf"
fig.savefig(out, bbox_inches="tight")

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

cold50_cmip = DA_all.sel(area=slice(0, 0.5)).mean(dim="area")
warm50_cmip = DA_all.sel(area=slice(0.5, 1)).mean(dim="area")
cold50_cmip_trim = cold50_cmip * 10 * 12
warm50_cmip_trim = warm50_cmip * 10 * 12

sources = ['ERA5','JRA55-do','uas & vas trend','uas & vas mean & trend','ERA5 winds','rsds & rlds trend','huss & tas trend','all-ish trends','NOAA OISST']
colors  = ['darkviolet','firebrick','cyan','b','violet','orange','limegreen','red','k']
cmap = dict(zip(sources, colors))

warm50_obs_da = xr.DataArray(
    np.array([
        warm50_era5.values,
        warm50_jra55.values,
        warm50_perturb_uas_vas.values,
        warm50_perturb_winds_mean_trend.values,
        warm50_perturb_winds.values,
        warm50_perturb_rsds_rlds.values,
        warm50_perturb_huss_tas.values,
        warm50_perturb_all.values,
        warm50_noaa.values
    ]) * 10 * 12,
    dims="source",
    coords={"source": sources},
    name="warm50_trend",
)

cold50_obs_da = xr.DataArray(
    np.array([
        cold50_era5.values,
        cold50_jra55.values,
        cold50_perturb_uas_vas.values,
        cold_50_perturb_winds_mean_trend.values,
        cold50_perturb_winds.values,
        cold50_perturb_rsds_rlds.values,
        cold50_perturb_huss_tas.values,
        cold50_perturb_all.values,
        cold50_noaa.values
    ]) * 10 * 12,
    dims="source",
    coords={"source": sources},
    name="cold50_trend",
)

keymap = {
    "ERA5": "era5_percentiles",
    "JRA55-do": "jra55_percentiles",
    "uas & vas trend": "uas_vas_trend_percentiles",
    "uas & vas mean & trend": "uas_vas_mean_trend_percentiles",
    "ERA5 winds": "era5_winds_percentiles",
    "rsds & rlds trend": "rsds_rlds_trend_percentiles",
    "huss & tas trend": "huss_tas_trend_percentiles",
    "all-ish trends": "all_trend_percentiles",
    "NOAA OISST": "noaa",
}

warm_means = dict(warm_means)
cold_means = dict(cold_means)
warm_means["noaa"] = noaa_warm50_mean
cold_means["noaa"] = noaa_cold50_mean

warm50_mean_da = xr.DataArray(
    np.array([np.asarray(warm_means[keymap[s]].values).squeeze().item() for s in sources]),
    dims="source",
    coords={"source": sources},
    name="warm50_mean",
)

cold50_mean_da = xr.DataArray(
    np.array([np.asarray(cold_means[keymap[s]].values).squeeze().item() for s in sources]),
    dims="source",
    coords={"source": sources},
    name="cold50_mean",
)

vals_warm50_omip = [float(percentiles_omip_trend[m].sel(area=slice(0.5, 1)).mean(dim="area").values * 10 * 12) for m in mod_labs]
vals_cold50_omip = [float(percentiles_omip_trend[m].sel(area=slice(0, 0.5)).mean(dim="area").values * 10 * 12) for m in mod_labs]

vals_warm50_omip_mean = [float(((dict_percentiles_omip[m].sel(area=slice(0.5, 1)).mean(dim="area")).mean(dim="time")).values) for m in mod_labs]
vals_cold50_omip_mean = [float(((dict_percentiles_omip[m].sel(area=slice(0, 0.5)).mean(dim="area")).mean(dim="time")).values) for m in mod_labs]

fig = plt.figure(figsize=(14, 12))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.35, wspace=0.25)

ax1 = fig.add_subplot(gs[1, 0])
ax2 = fig.add_subplot(gs[1, 1])

pos1 = ax1.get_position()
pos2 = ax2.get_position()

w = pos1.width
h = pos1.height
y_top = pos1.y0 + pos1.height + 0.08

x0 = (pos1.x0 + pos2.x1) / 2 - w / 2
ax0 = fig.add_axes([x0, y_top, w, h])

gap = 0.01
avail = pos2.x1 - (x0 + w) - gap
leg_w = max(0.06, min(0.20, avail))
axL = fig.add_axes([x0 + w + gap, y_top, leg_w, h])
axL.axis("off")

ax0.set_xlabel("Trend of Coldest 50% (°C/10yr)")
ax0.set_ylabel("Trend of Warmest 50% (°C/10yr)")


ax0.scatter(cold50_cmip_trim, warm50_cmip_trim, color="red", alpha=0.15)
ax0.scatter(np.nanmean(cold50_cmip_trim), np.nanmean(warm50_cmip_trim),
            color="indianred", edgecolors="k", s=140, linewidth=0.8, zorder=6)
ax0.scatter(vals_cold50_omip, vals_warm50_omip, color="slateblue", alpha=0.5, s=90)

for s, c in zip(sources, colors):
    ax0.scatter(cold50_obs_da.sel(source=s).item(), warm50_obs_da.sel(source=s).item(),
                color=c, s=90, linewidth=0.6, zorder=5)

for s in ["ERA5", "JRA55-do", "NOAA OISST"]:
    ax0.axvline(cold50_obs_da.sel(source=s).item(), linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
    ax0.axhline(warm50_obs_da.sel(source=s).item(), linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)

ax0.plot([-1, 1.25], [-1, 1.25], linestyle="--", color="k")
ax0.set_xlim([-0.1, 0.4])
ax0.set_ylim([-0.05, 0.45])

ax1.scatter(vals_cold50_omip, vals_cold50_omip_mean, color="slateblue", alpha=0.5, s=90)
for s, c in zip(sources, colors):
    ax1.scatter(cold50_obs_da.sel(source=s).item(), cold50_mean_da.sel(source=s).item(),
                color=c, s=90, linewidth=0.6, zorder=5)
for s in ["ERA5", "JRA55-do", "NOAA OISST"]:
    ax1.axvline(cold50_obs_da.sel(source=s).item(), linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
    ax1.axhline(cold50_mean_da.sel(source=s).item(), linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
ax1.set_xlabel(r"Trend of Coldest 50% ($^{\circ}$C/10yr)")
ax1.set_ylabel(r"Mean of Coldest 50% ($^{\circ}$C)")

ax2.scatter(vals_warm50_omip, vals_warm50_omip_mean, color="slateblue", alpha=0.5, s=90)
for s, c in zip(sources, colors):
    ax2.scatter(warm50_obs_da.sel(source=s).item(), warm50_mean_da.sel(source=s).item(),
                color=c, s=90, linewidth=0.6, zorder=5)
for s in ["ERA5", "JRA55-do", "NOAA OISST"]:
    ax2.axvline(warm50_obs_da.sel(source=s).item(), linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
    ax2.axhline(warm50_mean_da.sel(source=s).item(), linestyle="--", linewidth=1, alpha=0.8, color=cmap[s], zorder=1)
ax2.set_xlabel(r"Trend of Warmest 50% ($^{\circ}$C/10yr)")
ax2.set_ylabel(r"Mean of Warmest 50% ($^{\circ}$C)")

handles0 = [
    Line2D([0], [0], marker="o", linestyle="None", markersize=8,
           markerfacecolor="red", markeredgecolor="none", alpha=0.15, label="CMIP"),
    Line2D([0], [0], marker="o", linestyle="None", markersize=10,
           markerfacecolor="indianred", markeredgecolor="k", label="CMIP mean"),
    Line2D([0], [0], marker="o", linestyle="None", markersize=8,
           markerfacecolor="slateblue", markeredgecolor="none", alpha=0.5, label="OMIP"),
    Line2D([0], [0], linestyle="--", color="k", label="1–1 ratio"),
]
handles_sources = [
    Line2D([0], [0], marker="o", linestyle="None", markersize=8,
           markerfacecolor=c, markeredgecolor="none", label=s)
    for s, c in zip(sources, colors)
]

axL.legend(handles=handles0 + handles_sources, frameon=False, loc="center left")

plt.show()
out = "/Users/isabelaconde/Desktop/warm_cold_threepanel.pdf"
fig.savefig(out, bbox_inches="tight")

In [ ]:
# load gridded data processed in gadi 
from datetime import datetime


mod_labs = ['MRI-ESM2-0', 'FGOALS-f3-L','EC-EARTH3','ACCESS-OM2-JRA55-cycle6_iaf','ACCESS-OM2-ERA5']

save_directory = '/Users/isabelaconde/Desktop/data/omip/tos/'

file_paths = [f"{save_directory}tos_global_{model}_1982-2018_gr.nc" for model in mod_labs]
gr_omip_mean = {}

td = datetime.strptime("1982-01-01", "%Y-%m-%d")
k = 444
d = pd.date_range(td, periods=k, freq='MS')


for i, path in enumerate(file_paths):
    ds = xr.open_dataarray(path)
    ds = ds.assign_coords({'time': d})
    gr_omip_mean[mod_labs[i]] = xr.open_dataarray(path).mean(dim = 'time')

# gr_omip['MRI-ESM2-0']

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(17, 8))
axes = axes.flatten()  # Make it 1D so you can index with i
labels = ['MRI-ESM2-0', 'FGOALS-f3-L','EC-EARTH3','ACCESS-OM2-JRA55-cycle6','ACCESS-OM2-ERA5','NOAA-GFDL']
for i, da in enumerate(gr_omip_mean):
    im = (gr_omip_mean[da].sel(lon = slice(110,290), lat = slice(-20,20))).plot(ax=axes[i], add_colorbar=False, cmap='RdBu_r', vmin=16, vmax=30)
    print(gr_omip_mean[da].sel(lon = slice(110,290), lat = slice(-20,20)).mean(dim = ['lat','lon']).values)
    axes[i].set_title(labels[i])
        # Add colorbar only to axes[1] and axes[4] (i.e. subplots 2 and 5)
    if i in [2, 5]:
        cbar = fig.colorbar(im, ax=axes[i], orientation='vertical')

In [ ]:
for i, da in enumerate(gr_omip_mean):
    print(da)
